<font size="+3"><mark>Create the data set: time series, meta data</mark></font>

What this notebook does:
1. Creates a new `armCODA` folder with the `dataset` and `dataset_metadata` subfolders.
2. Copies all the time series npy files and renames them in `armCODA/dataset`.
   1. The original npy files must be in `../Data_Smartarm_MC/smartarm_gapfree/`
3. Convert the npy time series into csv files in `armCODA/dataset`.
4. Creates the clean `subjects_characteristics.csv` and `movements_annotations.csv` files in `armCODA/dataset_metadata`, from the following:
   1. `../Data_Smartarm_MC/Segmentation.xlsx`
   2. `../Data_Smartarm_MC/Tableau Patients SMARTARM.xlsx`
5. Creates the json metadata files per time series in in `armCODA/dataset` (from the clean `subjects_characteristics.csv` and `movements_annotations.csv` files).

In a nutshell, it creats the armCODA data set:
- `armCODA/dataset`: all 240 npy files, all 240 csv files, all 240 json files
- `armCODA/dataset_metadata`: `subjects_characteristics.csv` and `movements_annotations.csv`

TODO:
- upload on the nextcould

# Introduction

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import json
import os
from tqdm.notebook import tqdm
from pathlib import Path

from utils import create_path

In [2]:
cwd = Path.cwd()

In [44]:
def create_path(path):
    """Create path to be able to export files, figures, etc.
    """

    # Get the complete list of parent paths (including the wanted final path)
    list_parent_paths = [path] + list(path.parents)

    # Get the list of paths that do not exist, thus need to be created
    parent_path_to_create = []
    for path in list_parent_paths:
        if not(path.exists()):
            parent_path_to_create.append(path)
    parent_path_to_create = parent_path_to_create[::-1]

    # Create the (needed) paths
    for path in parent_path_to_create:
        if not(path.exists()):
            path.mkdir(exist_ok=True)

# Configuration dictionaries: new subject IDs and new movement IDs

## Movement IDs

In [3]:
# Movements' description and their abbreviations (taken from the demo's DDL)
d_mvt_desc_abb = {
    "Sagittal arm elevation, bilateral, seated" : "AEBPSA", 
    "Scapular arm elevation, bilateral, seated" : "AEBPSC", 
    "Frontal arm elevation, bilateral, seated" : "AEBPF", 
    "Scapular arm elevation with wrist rotation, bilateral, seated" : "AEBPSCR", 
    "Lateral elbow rotation, bilateral, seated" : "ABR",
    "Hair combing, right arm, seated" : "AMOD",
    "Hair combing, left arm, seated" : "AMOG", 
    "Low back washing, right arm, seated" : "AMFD",
    "Low back washing, left arm, seated" : "AMFG",
    "Sagittal arm elevation, bilateral, standing" : "DEBPSA",
    "Scapular arm elevation, bilateral, standing" : "DEBPSC",
    "Sagittal arm elevation, right arm, standing" : "DEBPSAD",
    "Sagittal arm elevation, left arm, standing" : "DEBPSAG",
    "Scapular arm elevation, right arm, standing" : "DEBPSCD",
    "Scapular arm elevation, left arm, standing" : "DEBPSCG",
}
display(d_mvt_desc_abb)

{'Sagittal arm elevation, bilateral, seated': 'AEBPSA',
 'Scapular arm elevation, bilateral, seated': 'AEBPSC',
 'Frontal arm elevation, bilateral, seated': 'AEBPF',
 'Scapular arm elevation with wrist rotation, bilateral, seated': 'AEBPSCR',
 'Lateral elbow rotation, bilateral, seated': 'ABR',
 'Hair combing, right arm, seated': 'AMOD',
 'Hair combing, left arm, seated': 'AMOG',
 'Low back washing, right arm, seated': 'AMFD',
 'Low back washing, left arm, seated': 'AMFG',
 'Sagittal arm elevation, bilateral, standing': 'DEBPSA',
 'Scapular arm elevation, bilateral, standing': 'DEBPSC',
 'Sagittal arm elevation, right arm, standing': 'DEBPSAD',
 'Sagittal arm elevation, left arm, standing': 'DEBPSAG',
 'Scapular arm elevation, right arm, standing': 'DEBPSCD',
 'Scapular arm elevation, left arm, standing': 'DEBPSCG'}

In [4]:
d_mvt_abb_id = dict()
for i, (_, value) in enumerate(d_mvt_desc_abb.items()):
    d_mvt_abb_id[value] = i
display(d_mvt_abb_id)

{'AEBPSA': 0,
 'AEBPSC': 1,
 'AEBPF': 2,
 'AEBPSCR': 3,
 'ABR': 4,
 'AMOD': 5,
 'AMOG': 6,
 'AMFD': 7,
 'AMFG': 8,
 'DEBPSA': 9,
 'DEBPSC': 10,
 'DEBPSAD': 11,
 'DEBPSAG': 12,
 'DEBPSCD': 13,
 'DEBPSCG': 14}

In [5]:
d_mvt_id_abb = dict()
for key, value in d_mvt_abb_id.items():
    d_mvt_id_abb[value] = key
display(d_mvt_id_abb)

{0: 'AEBPSA',
 1: 'AEBPSC',
 2: 'AEBPF',
 3: 'AEBPSCR',
 4: 'ABR',
 5: 'AMOD',
 6: 'AMOG',
 7: 'AMFD',
 8: 'AMFG',
 9: 'DEBPSA',
 10: 'DEBPSC',
 11: 'DEBPSAD',
 12: 'DEBPSAG',
 13: 'DEBPSCD',
 14: 'DEBPSCG'}

## Subject IDs

In [6]:
path_npy = Path("../Data_Smartarm_MC/smartarm_gapfree/")
npy_files = list(path_npy.rglob("*.npy"))

l_info = list()
for npy_file in sorted(npy_files):
    str_split = str(npy_file).split("smartarm_")[-1].split("_")
    subject_id = str(str_split[0])
    movement_id = str(str_split[1].split(".npy")[0])
    signal_npy = np.load(npy_file)
    shape0, shape1, shape2 = signal_npy.shape
    l_info.append([subject_id, movement_id, shape0, shape1, shape2])

df_timeseries_metadata = pd.DataFrame(l_info)
df_timeseries_metadata.columns = [
    "subject_id_str", "movement_abb", "n_sensors", "n_samples", "n_dim"
]
df_timeseries_metadata.insert(
    loc=0,
    column="subject_id",
    value=df_timeseries_metadata["subject_id_str"].astype("int")
)
df_timeseries_metadata.head()

,subject_id,subject_id_str,movement_abb,n_sensors,n_samples,n_dim
0,4,04,ABR,34,4001,3
1,4,04,AEBPF,34,4001,3
2,4,04,AEBPSA,34,4001,3
3,4,04,AEBPSC,34,4001,3
4,4,04,AEBPSCR,34,4001,3


In [7]:
l_unique_old_subject_ids = sorted(df_timeseries_metadata["subject_id"].unique().tolist())
l_unique_old_subject_ids

[4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [8]:
first_subject = min(l_unique_old_subject_ids)
n_subjects = len(l_unique_old_subject_ids)
d_subject_ids = {x: x - first_subject for x in range(first_subject, first_subject+n_subjects)}
display(d_subject_ids)

{4: 0,
 5: 1,
 6: 2,
 7: 3,
 8: 4,
 9: 5,
 10: 6,
 11: 7,
 12: 8,
 13: 9,
 14: 10,
 15: 11,
 16: 12,
 17: 13,
 18: 14,
 19: 15}

## Create an empty `armCODA` folder, copy the `npy` files there, and rename them

In [9]:
# Create the new folders
create_path(Path(cwd / "armCODA/dataset"))
create_path(Path(cwd / "armCODA/dataset_metadata"))

In [10]:
# Copy all npy files in the `armCODA/dataset` folder
# bash commands compatible on macOS or Linux
! cp ../Data_Smartarm_MC/smartarm_gapfree/*.npy armCODA/dataset

In [11]:
# Get all the npy files
folder_npy = Path(cwd / "armCODA/dataset")
npy_files = sorted(list(folder_npy.rglob("smartarm_*.npy")))
print(len(npy_files))

240


In [12]:
# Rename all the npy files
for old_path_file in npy_files:
    filename = old_path_file.name
    filename_split = filename.split("_")
    subject_id, movement_abb_str = int(filename_split[1]), filename_split[2].split(".npy")[0]
    new_subject_id = d_subject_ids[subject_id]
    new_movement_id = d_mvt_abb_id[movement_abb_str]
    new_filename = f"armcoda_subject{new_subject_id}_movement{new_movement_id}.npy"
    new_path_file = folder_npy / Path(new_filename)
    old_path_file.rename(new_path_file)

# Convert the 3D `npy` time series files into 2D `csv` files

In [14]:
new_npy_files = sorted(list(folder_npy.rglob("armcoda_*.npy")))
print(len(new_npy_files))

240


In [18]:
for npy_path in tqdm(new_npy_files):
    filename = npy_path.name
    if '.npy' in filename:
        df_res = pd.DataFrame()
        ts = np.load(folder_npy / filename)
        for marker in range(len(ts)):
            df_res['marker_{}_x'.format(marker)] = ts[marker][:, 0]
            df_res['marker_{}_y'.format(marker)] = ts[marker][:, 1]
            df_res['marker_{}_z'.format(marker)] = ts[marker][:, 2]
        df_res.to_csv(folder_npy / filename.replace('.npy', '.csv'), index=False)

  0%|          | 0/240 [00:00<?, ?it/s]

/var/folders/cf/3dw4z3cd51371v3k7m1m3w600000gn/T/ipykernel_95690/1880506647.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_res['marker_{}_y'.format(marker)] = ts[marker][:, 1]
/var/folders/cf/3dw4z3cd51371v3k7m1m3w600000gn/T/ipykernel_95690/1880506647.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_res['marker_{}_z'.format(marker)] = ts[marker][:, 2]
/var/folders/cf/3dw4z3cd51371v3k7m1m3w600000gn/T/ipykernel_95690/1880506647.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of c

In [19]:
# Sanity check
sensor_ID = 4
print(np.allclose(df_res[f'marker_{sensor_ID}_x'].to_numpy(), ts[sensor_ID][:, 0]))
print(np.allclose(df_res[f'marker_{sensor_ID}_y'].to_numpy(), ts[sensor_ID][:, 1]))
print(np.allclose(df_res[f'marker_{sensor_ID}_z'].to_numpy(), ts[sensor_ID][:, 2]))

True
True
True


# Create the clean movements iterations' annotations file

In [20]:
# Load the data
# Rename the features
# Drop rows where the annotation is not available for the first two iterations.
rename_columns_movements = {
    "sujet":"subject_ID",
    "movement":"movement_ID",
}
df_movements_annotations = (
    pd.read_excel('../Data_Smartarm_MC/Segmentation.xlsx')
    [["sujet", "movement", "s1", "e1", "s2", "e2", "s3", "e3"]]
    .rename(columns=rename_columns_movements)
    .dropna(subset=["s1", "e1", "s2", "e2"])
)

# Convert to numeric
df_movements_annotations["s3"] = pd.to_numeric(df_movements_annotations["s3"], errors='coerce')
df_movements_annotations["e3"] = pd.to_numeric(df_movements_annotations["e3"], errors='coerce')

# Sanity check: the start must be before the end
for i in range(1, 4):
    df_movements_annotations.loc[df_movements_annotations[f"s{i}"] > df_movements_annotations[f"e{i}"], f"e{i}"] = np.nan
    cond = (df_movements_annotations[f"s{i}"] > df_movements_annotations[f"e{i}"]).sum() == 0
    err_msg = (
        f"For the iteration #{i}, we have at least occurence where the"
        " start is after the end."
    )
    assert cond, err_msg

# Rename subject IDs
df_movements_annotations["subject_ID"] = df_movements_annotations["subject_ID"].replace(d_subject_ids)

# Rename movements IDs
df_movements_annotations["movement_ID"] = df_movements_annotations["movement_ID"].replace(d_mvt_abb_id)

# Set the type
change_type_movements = {
    "subject_ID":"int",
    "movement_ID":"int",
    "s1":"int",
    "e1":"int",
    "s2":"int",
    "e2":"int",
}
df_movements_annotations = df_movements_annotations.astype(change_type_movements)

# Round to avoid display of unncessary decimals
df_movements_annotations = df_movements_annotations.round(decimals=0)

# Display
df_movements_annotations.head()

,subject_ID,movement_ID,s1,e1,s2,e2,s3,e3
15,0,0,211,1357,1502,2892,2912.0,NaN
16,0,1,121,1357,1435,2634,2791.0,3990.0
17,0,2,113,1353,1534,2763,2930.0,NaN
18,0,3,246,1592,1697,3160,3275.0,NaN
19,0,4,313,1186,1303,3076,3255.0,NaN


In [21]:
df_movements_annotations.dtypes

subject_ID       int64
movement_ID      int64
s1               int64
e1               int64
s2               int64
e2               int64
s3             float64
e3             float64
dtype: object

In [22]:
df_movements_annotations.isna().sum()

subject_ID       0
movement_ID      0
s1               0
e1               0
s2               0
e2               0
s3               7
e3             161
dtype: int64

In [23]:
subject_ids = df_movements_annotations["subject_ID"].unique().tolist()
print(f"{subject_ids = }")
print(f"{len(subject_ids) = }")

subject_ids = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
len(subject_ids) = 16


In [25]:
df_movements_annotations.to_csv("armCODA/dataset_metadata/movements_annotations.csv", index=False)

# Create the clean patients characteristics' file

In [26]:
# Newly found laterality of the patients
more_laterality = {
    5:"L",
    6:"R",
    14:"R",
}

# Rename the columns
rename_columns_subjects = {
    "Sujet":"subject_ID",
    "Sexe":"gender",
    "AGE":"age",
    "TAILLE (cm)":"height_cm",
    "POIDS (kg)":"weight_kg",
    "LATERALITE":"laterality",
}

# Load the data with the desired columns and rename them
df_subjects_characteristics = (
    pd.read_excel('../Data_Smartarm_MC/Tableau Patients SMARTARM.xlsx')
    .drop(columns=["NOM", "PRENOM", "BMI"])
    .rename(columns=rename_columns_subjects)
)

# Get the patients who have a sufficient movement annotation
df_subjects_characteristics = df_subjects_characteristics.query(f"subject_ID in {l_unique_old_subject_ids}").reset_index(drop=True)

# Add the newly found lateralities
for subject_id in more_laterality.keys():
    df_subjects_characteristics.loc[df_subjects_characteristics["subject_ID"] == subject_id, "laterality"] = more_laterality[subject_id]

# Convert the laterality results into the English format
df_subjects_characteristics["laterality"] = df_subjects_characteristics["laterality"].replace({"D":"R", "G":"L"})

# Compute the BMI and round it
df_subjects_characteristics.insert(
    loc=5,
    column="BMI",
    value=df_subjects_characteristics["weight_kg"] / (df_subjects_characteristics["height_cm"]*0.01)**2,
)
df_subjects_characteristics = df_subjects_characteristics.round(decimals=1)

# Reset the subject IDs so that they start at 0
df_subjects_characteristics["subject_ID"] = df_subjects_characteristics["subject_ID"].replace(d_subject_ids)

# Display
display(df_subjects_characteristics)

,subject_ID,gender,age,height_cm,weight_kg,BMI,laterality
0,0,M,30,175,77.0,25.1,NaN
1,1,M,28,179,73.0,22.8,L
2,2,M,27,188,78.0,22.1,R
3,3,F,50,172,68.0,23.0,R
4,4,F,52,156,64.0,26.3,NaN
5,5,F,62,158,51.0,20.4,R
6,6,F,65,168,74.0,26.2,NaN
7,7,M,57,173,75.0,25.1,NaN
8,8,M,42,175,84.5,27.6,NaN
9,9,M,40,181,95.0,29.0,NaN


In [27]:
df_subjects_characteristics.isna().sum()

subject_ID    0
gender        0
age           0
height_cm     0
weight_kg     0
BMI           0
laterality    8
dtype: int64

In [28]:
df_subjects_characteristics.dtypes

subject_ID      int64
gender         object
age             int64
height_cm       int64
weight_kg     float64
BMI           float64
laterality     object
dtype: object

In [29]:
df_subjects_characteristics.to_csv("armCODA/dataset_metadata/subjects_characteristics.csv", index=False)

# Create the `json` meta data files

In [37]:
def get_mov_name(movement_ID):

    name = d_mvt_id_abb[movement_ID]

    mov_name = {
        'movement_ID':movement_ID,
        'Position':None,
        'Movement':{'Limb':None,'Laterality':None,'Action':None},
        'Plan':None,
        'Hand':{'Laterality':None,'Position':None},
    }
    
    if name == 'ABR':
        mov_name['Movement']['Limb'] = 'Arm'
        mov_name['Movement']['Action'] = 'Rotation'
    elif name == 'AEBPSCR':
        mov_name['Position'] = 'Seat'
        mov_name['Movement']['Limb'] = ['Arm', 'Wrist']
        mov_name['Movement']['Action'] = ['Elevation', 'Rotation']
        mov_name['Plan'] == 'Scapular'
    elif name[0] == 'D' or name[0:2] == 'AE':
        if name[0] == 'D':
            mov_name['Position'] = 'Stand'
        else:
            mov_name['Position'] = 'Seat'
        
        mov_name['Movement']['Limb'] = 'Arm'
        mov_name['Movement']['Action'] = 'Elevation'
        
        if name[-1] == 'D':
            mov_name['Movement']['Laterality'] = 'Right'
        elif name[-1] == 'G':
            mov_name['Movement']['Laterality'] = 'Left'
        else:
            mov_name['Movement']['Laterality'] = 'Bilateral'
        
        if name[4] == 'F':
            mov_name['Plan'] = 'Frontal'
        else:
            if name[5] == 'C':
                mov_name['Plan'] = 'Scapular'
            elif name[5] == 'A':
                mov_name['Plan'] = 'Sagittal'
            
    elif name[0:2] == 'AM':
        mov_name['Position'] = 'Seat'

        if name[3] == 'D':
            mov_name['Hand']['Laterality'] = 'Right'
        elif name[3] == 'G':
            mov_name['Hand']['Laterality'] = 'Left'

        if name[2] == 'O':
            mov_name['Hand']['Position'] = 'Head'
        elif name[2] == 'F':
            mov_name['Hand']['Position'] = 'Bottom'
    
    return mov_name
    
    
def get_patient_info(subject_ID, df_subjects_characteristics):
    patient_dict = {
        'ID':None,
        'Gender':None,
        'Age':None,
        'Height (cm)':None,
        'Weight (kg)':None,
        'BMI (kg/m^2)':None,
        'Laterality':None,
    }
    
    patient_dict['ID'] = subject_ID
    
    if subject_ID not in df_subjects_characteristics["subject_ID"].tolist():
        return patient_dict
    
    patient_dict['Gender'] = df_subjects_characteristics.at[subject_ID, 'gender']
    patient_dict['Age'] = int(df_subjects_characteristics.at[subject_ID, 'age'])
    patient_dict['Height (cm)'] = float(df_subjects_characteristics.at[subject_ID, 'height_cm'])
    patient_dict['Weight (kg)'] = float(df_subjects_characteristics.at[subject_ID, 'weight_kg'])
    patient_dict['BMI (kg/m^2)'] = round(float(df_subjects_characteristics.at[subject_ID, 'BMI']), 1)
    patient_dict['Laterality'] = df_subjects_characteristics.at[subject_ID, 'laterality']
    
    return patient_dict


def get_ts_event(subject_ID, movement_ID, df_movements_annotations):
    dict_gesture = {
        'Iteration_1': None,
        'Iteration_2': None,
        'Iteration_3': None
    }
    df_temp = df_movements_annotations.query(f"subject_ID=={subject_ID} & movement_ID == {movement_ID}")
    
    dict_gesture['Iteration_1'] = [int(df_temp['s1'].values[0]), int(df_temp['e1'].values[0])]
    dict_gesture['Iteration_2'] = [int(df_temp['s2'].values[0]), int(df_temp['e2'].values[0])]
    dict_gesture['Iteration_3'] = [df_temp['s3'].values[0], df_temp['e3'].values[0]]

    return dict_gesture
        
    

# df_segmentation  = pd.read_excel('../Data_Smartarm_MC/Segmentation.xlsx', index_col=0)
# info_df = pd.read_csv('../Data_Smartarm_MC/df_smartarm.csv', index_col=0)
# df_patient = pd.read_excel('../Data_Smartarm_MC/Tableau Patients SMARTARM.xlsx', index_col=0)


In [38]:
npy_files = list(folder_npy.rglob("armcoda_*.npy"))
len(npy_files)

240

In [39]:
for npy_file_path in tqdm(npy_files):
    filename = npy_file_path.name
    filename_split = filename.split("_")
    subject_ID = int(filename_split[1].split("subject")[1])
    movement_ID = int(filename_split[2].split(".npy")[0].split("movement")[1])
    
    dict_gesture = get_ts_event(subject_ID, movement_ID, df_movements_annotations)
    mov_dict = get_mov_name(movement_ID)
    patient_dict = get_patient_info(subject_ID, df_subjects_characteristics)

    to_save = {
        'Data_filename':filename,
        'Patient_info':patient_dict,
        'Movement_info':mov_dict,
        'Movement_label':dict_gesture
    }

    with open(folder_npy / '{}.json'.format(filename.split('.')[0]), 'w') as f:
        json.dump(to_save, f, indent=4)

  0%|          | 0/240 [00:00<?, ?it/s]

# Sanity checks

In [40]:
!cat armCODA/dataset_metadata/subjects_characteristics.csv

subject_ID,gender,age,height_cm,weight_kg,BMI,laterality
0,M,30,175,77.0,25.1,
1,M,28,179,73.0,22.8,L
2,M,27,188,78.0,22.1,R
3,F,50,172,68.0,23.0,R
4,F,52,156,64.0,26.3,
5,F,62,158,51.0,20.4,R
6,F,65,168,74.0,26.2,
7,M,57,173,75.0,25.1,
8,M,42,175,84.5,27.6,
9,M,40,181,95.0,29.0,
10,M,47,180,62.0,19.1,R
11,M,62,171,84.0,28.7,
12,M,50,178,75.0,23.7,R
13,F,47,170,65.0,22.5,
14,M,23,183,93.0,27.8,R
15,M,25,172,61.0,20.6,R


In [41]:
with open(folder_npy / 'armcoda_subject0_movement0.json') as f:
    metadata = json.load(f)
metadata

{'Data_filename': 'armcoda_subject0_movement0.npy',
 'Patient_info': {'ID': 0,
  'Gender': 'M',
  'Age': 30,
  'Height (cm)': 175.0,
  'Weight (kg)': 77.0,
  'BMI (kg/m^2)': 25.1,
  'Laterality': nan},
 'Movement_info': {'movement_ID': 0,
  'Position': 'Seat',
  'Movement': {'Limb': 'Arm',
   'Laterality': 'Bilateral',
   'Action': 'Elevation'},
  'Plan': 'Sagittal',
  'Hand': {'Laterality': None, 'Position': None}},
 'Movement_label': {'Iteration_1': [211, 1357],
  'Iteration_2': [1502, 2892],
  'Iteration_3': [2912.0, nan]}}